<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/931_IRMv3_Nodes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IRMv3 Orchestrator Nodes

## Architecture Review & Operational Explanation

The `nodes.py` module defines the **execution flow of the Integration & Risk Management Orchestrator**.

This file does not contain business logic itself.

Instead, it performs a much more important role:

**It coordinates the execution of the orchestrator pipeline.**

Each node represents a **stage in the operational intelligence workflow**.

The nodes call specialized utility functions that perform:

* data ingestion
* scoring
* risk evaluation
* reporting

This design keeps the orchestration layer **clean, predictable, and easy to reason about**.

---

# Overall Execution Flow

The orchestrator follows a simple and transparent workflow:

```text
Goal → Planning → Data Loading → Scoring → Rollups & Triggers → Executive Report
```

Each stage performs a specific function.

| Node              | Purpose                            |
| ----------------- | ---------------------------------- |
| goal_node         | Defines the purpose of the run     |
| planning_node     | Defines the execution steps        |
| data_loading_node | Loads telemetry and builds lookups |
| scoring_node      | Computes operational health scores |
| report_node       | Produces the executive report      |

This flow mirrors how **real operational monitoring systems work**.

---

# Goal Node

The goal node defines the purpose of the orchestrator run.

```python
goal_node()
```

It sets the system objective:

> Evaluate integration health, dependency exposure, governance responsiveness, and automation value.

This node provides **context for the rest of the workflow**.

Although simple, this step is important for transparency.

Anyone reviewing the system can immediately see:

**what this orchestrator is intended to accomplish.**

---

# Planning Node

The planning node defines the **execution sequence**.

```python
planning_node()
```

The plan outlines four stages:

1. Load operational data
2. Compute ecosystem health scores
3. Identify operational triggers
4. Generate an executive report

This structure is intentionally simple.

The system avoids dynamic planning or AI-based task selection.

Instead, the plan remains **deterministic and predictable**.

This is important for systems designed to monitor operational risk.

Leadership needs monitoring systems that behave **consistently and reliably**.

---

# Data Loading Node

The `make_data_loading_node()` function creates a node responsible for ingesting all telemetry data.

This node performs three key actions:

1. Resolve the data directory relative to the project root
2. Load all IRM datasets using the loader utility
3. Populate the orchestrator state with structured data

The node collects telemetry for:

* agents
* workflows
* integrations
* dependency graphs
* execution logs
* risk signals
* remediation records

It also stores several derived structures:

```text
agents_lookup
workflows_by_agent
workflows_by_integration
```

These lookup maps make the scoring stage significantly more efficient.

### Why this design matters

By centralizing all data ingestion in a single node, the orchestrator guarantees that:

* scoring logic never needs to fetch external data
* every run uses a consistent dataset snapshot
* data lineage is preserved

This is essential for **reproducible operational analysis**.

---

# Scoring Node

The scoring node performs the **core analysis of the AI ecosystem**.

It calculates four operational health metrics:

| Score                     | Operational Meaning               |
| ------------------------- | --------------------------------- |
| Integration Stability     | Reliability of system connections |
| Dependency Exposure       | Architectural fragility           |
| Governance Responsiveness | Organizational issue resolution   |
| Automation Value          | ROI from automation               |

These scores are calculated using deterministic formulas defined in the scoring utilities.

The node then computes:

```text
portfolio_score
portfolio_status
```

This produces the **overall operational health signal**.

---

# Risk Signal Aggregation

The scoring node also aggregates governance signals from the dataset.

It calculates:

```text
open_risks
overdue_remediations
```

These values feed into the **executive trigger system**.

This ensures the orchestrator is not only evaluating technical conditions but also measuring **organizational responsiveness**.

---

# Executive Trigger Generation

Once scores and risk counts are calculated, the system generates escalation signals.

Triggers occur when thresholds are crossed, such as:

* integration reliability below acceptable levels
* fragile dependency architecture
* excessive open risks
* overdue remediation actions

These triggers convert operational metrics into **clear governance signals for leadership**.

---

# Report Node

The final node generates the **executive report**.

The report includes:

* portfolio health verdict
* score breakdown
* operational triggers
* system telemetry summary
* risk visibility
* remediation progress
* recommended next steps

The report is saved to disk using a timestamped filename.

Example output path:

```
agents/output/irm_v3_reports/irm_v3_report_YYYYMMDD_HHMMSS.md
```

This creates a permanent historical record of each evaluation.

Such records are valuable for:

* governance audits
* operational reviews
* trend analysis

---

# Why This Architecture Is Strong

Several design principles are clearly visible in this node structure.

### Thin orchestration layer

Nodes do not contain business logic.

All analysis is handled by specialized utility functions.

This separation keeps the orchestrator easy to maintain.

---

### Deterministic workflow

The pipeline is fixed and predictable.

There is no hidden decision logic inside the orchestration layer.

---

### State-driven execution

All nodes operate on the same shared state.

This ensures that data flows through the system transparently.

---

### Config-driven thresholds

Critical thresholds come from the configuration object.

This allows organizations to tune operational sensitivity without modifying code.

---

# Why Executives Would Trust This System

Executives responsible for AI operations need monitoring systems that are:

* predictable
* transparent
* auditable
* reliable

This orchestrator architecture supports all four.

Every run follows the same steps.

Every decision is traceable to explicit inputs.

Every report records the dataset used to generate conclusions.

This transparency builds **confidence in the system’s outputs**.

---

# What Makes This Different From Most AI Agents

Most AI agents perform **tasks or analysis**.

Examples include:

* generating content
* answering questions
* analyzing documents

IRMv3 performs a different function.

It acts as an **operational intelligence layer for AI systems themselves**.

Instead of automating business tasks, it monitors the **health, risk, and governance of the automation environment**.

This shift is important as organizations deploy more AI systems.

Without orchestration layers like this, companies risk building **complex automation ecosystems that no one is truly monitoring or governing**.

---

# Opportunities for Enhancement

Your node structure is already strong, but a few small improvements could enhance it further.

### Execution timing metrics

You could capture execution duration for each node.

This would allow the orchestrator to detect **performance degradation in its own monitoring pipeline**.

---

### Node-level error tracking

Capturing errors at the node level could improve observability.

---

### Trend signal generation

The scoring node could optionally compare current results with historical snapshots.

This would enable early detection of **system deterioration trends**.

---

# Overall Assessment

The node architecture completes the IRMv3 system.

Together with the loader, scoring system, rollups, and reporting modules, it creates a **fully structured operational intelligence pipeline**.

The orchestrator:

1. collects telemetry
2. evaluates operational health
3. detects risks
4. generates executive intelligence

This is exactly the type of architecture organizations will need as AI deployments scale.

Rather than focusing solely on automation capabilities, IRMv3 ensures that **the AI ecosystem itself remains observable, governable, and stable**.



In [ ]:
"""IRMv3 orchestrator nodes. Thin layer: orchestrate via utilities."""

from pathlib import Path
from typing import Any, Callable, Dict

from config import IRMv3Config, IRMv3State
from agents.irm_v3.orchestrator.utilities import (
    load_all_irm_data,
    compute_integration_stability_score,
    compute_dependency_exposure_score,
    compute_governance_responsiveness_score,
    compute_automation_value_score,
    compute_portfolio_score,
    compute_portfolio_status,
    build_score_rollup,
    build_executive_triggers,
    build_irm_report,
)


def goal_node(state: IRMv3State) -> Dict[str, Any]:
    """Set fixed goal for IRM evaluation run."""
    return {
        "goal": {
            "name": "AI Ecosystem Integration & Risk Assessment",
            "description": "Evaluate integration health, dependency exposure, governance responsiveness, and automation value; produce portfolio health signal and executive report.",
        }
    }


def planning_node(state: IRMv3State) -> Dict[str, Any]:
    """Fixed plan: load data → score → report."""
    return {
        "plan": [
            {"step": 1, "action": "Load all IRM data from data_dir"},
            {"step": 2, "action": "Compute four core scores and portfolio status"},
            {"step": 3, "action": "Build rollups and executive triggers"},
            {"step": 4, "action": "Generate executive report"},
        ]
    }


def make_data_loading_node(config: IRMv3Config, project_root: str) -> Callable[[IRMv3State], Dict[str, Any]]:
    """Load all IRM data; resolve data_dir against project_root."""

    def node(state: IRMv3State) -> Dict[str, Any]:
        data_dir = state.get("data_dir") or config.data_dir
        file_names = {
            "agents_file": config.agents_file,
            "workflows_file": config.workflows_file,
            "system_integrations_file": config.system_integrations_file,
            "dependency_graph_file": config.dependency_graph_file,
            "mission_runs_file": config.mission_runs_file,
            "task_execution_logs_file": config.task_execution_logs_file,
            "integration_health_logs_file": config.integration_health_logs_file,
            "dependency_incidents_file": config.dependency_incidents_file,
            "risk_signals_file": config.risk_signals_file,
            "remediation_actions_file": config.remediation_actions_file,
            "historical_snapshots_file": config.historical_snapshots_file,
            "governance_reviews_file": config.governance_reviews_file,
            "expected_vs_actual_value_file": config.expected_vs_actual_value_file,
            "kpis_cost_file": config.kpis_cost_file,
        }
        out = load_all_irm_data(data_dir, project_root=project_root, **file_names)

        return {
            "agents": out.get("agents", []),
            "workflows": out.get("workflows", []),
            "system_integrations": out.get("system_integrations", []),
            "dependency_graph": out.get("dependency_graph", []),
            "mission_runs": out.get("mission_runs", []),
            "task_execution_logs": out.get("task_execution_logs", []),
            "integration_health_logs": out.get("integration_health_logs", []),
            "dependency_incidents": out.get("dependency_incidents", []),
            "risk_signals": out.get("risk_signals", []),
            "remediation_actions": out.get("remediation_actions", []),
            "historical_snapshots": out.get("historical_snapshots", []),
            "governance_reviews": out.get("governance_reviews", []),
            "expected_vs_actual_value": out.get("expected_vs_actual_value", []),
            "kpis_cost": out.get("kpis_cost", []),
            "agents_lookup": out.get("agents_lookup", {}),
            "workflows_by_agent": out.get("workflows_by_agent", {}),
            "workflows_by_integration": out.get("workflows_by_integration", {}),
            "data_snapshot_loaded_at": out.get("data_snapshot_loaded_at"),
            "validation_warnings": out.get("validation_warnings", []),
            "loader_record_counts": out.get("loader_record_counts", {}),
        }
    return node


def make_scoring_node(config: IRMv3Config) -> Callable[[IRMv3State], Dict[str, Any]]:
    """Compute four scores, portfolio score, status, rollup, and executive triggers."""

    def node(state: IRMv3State) -> Dict[str, Any]:
        integration_stability = compute_integration_stability_score(
            state.get("integration_health_logs") or []
        )
        dependency_exposure = compute_dependency_exposure_score(
            state.get("dependency_graph") or [],
            state.get("dependency_incidents") or [],
            state.get("workflows_by_agent") or {},
            state.get("system_integrations") or [],
        )
        governance_responsiveness = compute_governance_responsiveness_score(
            state.get("remediation_actions") or [],
            state.get("risk_signals") or [],
        )
        automation_value = compute_automation_value_score(
            state.get("task_execution_logs") or [],
            state.get("workflows") or [],
            state.get("expected_vs_actual_value"),
            state.get("kpis_cost"),
        )

        portfolio_score = compute_portfolio_score(
            integration_stability,
            dependency_exposure,
            governance_responsiveness,
            automation_value,
            weights={
                "integration": config.weight_integration,
                "dependency": config.weight_dependency,
                "governance": config.weight_governance,
                "automation": config.weight_automation,
            },
        )
        portfolio_status = compute_portfolio_status(
            integration_stability,
            dependency_exposure,
            governance_responsiveness,
            automation_value,
            portfolio_score,
            critical_threshold=config.critical_threshold,
            at_risk_portfolio=config.at_risk_portfolio_score,
            at_risk_two_below=config.at_risk_two_scores_below,
        )

        score_rollup = build_score_rollup(
            integration_stability,
            dependency_exposure,
            governance_responsiveness,
            automation_value,
            portfolio_score,
            portfolio_status,
        )

        risk_signals = state.get("risk_signals") or []
        remediation_actions = state.get("remediation_actions") or []
        open_risks = sum(1 for r in risk_signals if (r.get("status") or "").lower() == "open")
        overdue = sum(1 for r in remediation_actions if r.get("missed_deadline_flag") is True)

        executive_triggers = build_executive_triggers(
            integration_stability,
            dependency_exposure,
            governance_responsiveness,
            automation_value,
            portfolio_status,
            open_risks,
            overdue,
            integration_critical=config.integration_stability_critical,
            dependency_critical=config.dependency_exposure_critical,
            governance_critical=config.governance_responsiveness_critical,
            automation_critical=config.automation_value_critical,
            open_risks_critical=config.open_risks_critical,
            overdue_remediations_critical=config.overdue_remediations_critical,
        )

        return {
            "integration_stability_score": integration_stability,
            "dependency_exposure_score": dependency_exposure,
            "governance_responsiveness_score": governance_responsiveness,
            "automation_value_score": automation_value,
            "portfolio_score": portfolio_score,
            "portfolio_status": portfolio_status,
            "score_rollup": score_rollup,
            "executive_triggers": executive_triggers,
        }
    return node


def make_report_node(config: IRMv3Config, project_root: str) -> Callable[[IRMv3State], Dict[str, Any]]:
    """Build markdown report and write to reports_dir."""

    def node(state: IRMv3State) -> Dict[str, Any]:
        reports_dir = state.get("reports_dir") or config.reports_dir
        base = Path(project_root) / reports_dir
        base.mkdir(parents=True, exist_ok=True)

        report_content = build_irm_report(
            portfolio_status=state.get("portfolio_status", "UNKNOWN"),
            portfolio_score=state.get("portfolio_score", 0),
            integration_stability=state.get("integration_stability_score", 0),
            dependency_exposure=state.get("dependency_exposure_score", 0),
            governance_responsiveness=state.get("governance_responsiveness_score", 0),
            automation_value=state.get("automation_value_score", 0),
            score_rollup=state.get("score_rollup", {}),
            executive_triggers=state.get("executive_triggers", []),
            loader_record_counts=state.get("loader_record_counts", {}),
            data_snapshot_loaded_at=state.get("data_snapshot_loaded_at"),
            validation_warnings=state.get("validation_warnings"),
            mission_runs=state.get("mission_runs"),
            risk_signals=state.get("risk_signals"),
            remediation_actions=state.get("remediation_actions"),
        )

        from datetime import datetime, timezone
        stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        report_path = base / f"irm_v3_report_{stamp}.md"
        report_path.write_text(report_content, encoding="utf-8")

        return {
            "mission_report": report_content,
            "report_file_path": str(report_path),
        }
    return node
